In [2]:
import torch
import numpy as np
import random
from minicons import scorer

MODEL = "Qwen/Qwen3-VL-2B-Instruct"
CACHE_DIR = "/mnt/dv/wid/projects3/Rogers-muri-human-ai/zstuddiford"
SEED = 0

# --- seed everything ---
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- load model ---
lm = scorer.VLMScorer(MODEL, device="cuda", torch_dtype=torch.bfloat16, cache_dir=CACHE_DIR)

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [137]:
import random, pandas as pd, re
random.seed(0)

# ============================================================
#  Correct pluralization: explicit irregular tables + rules.
#  Nothing is guessed — every exception is listed.
# ============================================================
IRREG_PL = {
    'mouse':'mice','goose':'geese','ox':'oxen','louse':'lice',
    'sheep':'sheep','deer':'deer','fish':'fish','bison':'bison','moose':'moose',
    'cod':'cod','trout':'trout','salmon':'salmon','carp':'carp','perch':'perch',
    'pike':'pike','bass':'bass','elk':'elk','swine':'swine','gnu':'gnu',
    'aircraft':'aircraft','offspring':'offspring',
}
F_TO_VES = {
    'wolf':'wolves','calf':'calves','half':'halves','loaf':'loaves','leaf':'leaves',
    'hoof':'hooves','shelf':'shelves','thief':'thieves','wharf':'wharves',
    'dwarf':'dwarves','scarf':'scarves','elf':'elves','self':'selves',
    'knife':'knives','life':'lives','wife':'wives',
}
def pluralize(w):
    if w in IRREG_PL: return IRREG_PL[w]
    if w in F_TO_VES: return F_TO_VES[w]
    if w.endswith(('s','x','z','sh','ch')):           return w + 'es'
    if w.endswith('y') and len(w) > 1 and w[-2] not in 'aeiou': return w[:-1] + 'ies'
    return w + 's'   # default (covers -o animals: hippo->hippos, koi->kois)

def make_pairs(words):
    """sg/pl dicts, deduped, dropping invariant-plural items (sg==pl) since
    they can't form a number minimal pair."""
    out, seen = [], set()
    for w in words:
        if w in seen: continue
        seen.add(w)
        pl = pluralize(w)
        if pl != w:
            out.append({'sg': w, 'pl': pl})
    return out


_SUBJ_SG = [
 'duck','dog','cat','bird','horse','fox','lion','frog','goat','bear','wolf','owl','seal','cow','pig',
 'hen','rat','crab','snail','moth','bee','ant','toad','hawk','crow','mole','newt','wasp','goose','mouse',
 'calf','lamb','colt','chick','cub','pup','kit','doe','ewe','ram','bull','mare','sow','drake','finch',
 'wren','lark','dove','gull','swan','heron','robin','sparrow','beetle','spider','lizard','otter','badger',
 'rabbit','hare','stoat','vole','shrew','weasel','ferret','mink','marten','raccoon','skunk','beaver',
 'squirrel','chipmunk','gopher','hamster','gerbil','hedgehog','porcupine','possum','koala','wombat','camel',
 'llama','donkey','mule','pony','zebra','antelope','gazelle','buffalo','bison','elk','panda','tiger',
 'leopard','cheetah','jaguar','cougar','lynx','bobcat','jackal','coyote','dingo','hyena','meerkat',
 'parrot','falcon','eagle','osprey','magpie','starling','pigeon','quail','pheasant','turkey','peacock',
 'flamingo','pelican','puffin','penguin','turtle','tortoise','gecko','iguana','salamander','minnow',
 'mantis','cricket','locust','hornet','beetle','weevil','aphid','earwig','firefly','ladybug',
 'dragonfly','grasshopper','caterpillar','centipede','millipede','tick','flea','gnat','midge','termite',
 'clam','oyster','mussel','prawn','shrimp','lobster','barnacle','urchin','jellyfish','starfish',
 'eel','trout','salmon','carp','perch','pike','bass','cod','herring','sardine',
 'shark','dolphin','whale','walrus','manatee','seahorse','stingray','octopus','squid','cuttlefish',
 'cobra','viper','python','adder','mamba','boa','rattler','skink','chameleon','monitor',
 'crane','stork','ibis','egret','plover','sandpiper','warbler','thrush','finch','bunting',
 'kestrel','harrier','buzzard','vulture','condor','raven','jackdaw','rook','jay','nuthatch',
 'mongoose','aardvark','pangolin','tapir','okapi','gnu','ibex','chamois','markhor','tahr',
 'wallaby','wombat','quokka','bandicoot','numbat','dunnart','quoll','potoroo','bilby','bettong',
 'ox','ape','yak','bat','ray','sow','boar','stag','deer','fawn','foal','joey','tern','kite',
 'slug','worm','grub','larva','moose','sloth','lemur','hippo','rhino','sheep','hound','puppy',
 'piglet','kitten','cock','boar','mare','colt','stallion','filly','heifer','steer','wether','hog',
 'gander','gosling','cygnet','duckling','fledgling','nestling','tadpole','minnow','guppy','koi',
]
SUBJECTS = make_pairs(_SUBJ_SG)

_DIST_SG = [
 'tree','car','box','wall','table','house','pond','bridge','gate','fence','rock','lamp','chair','shed',
 'barn','cart','bench','post','window','door','crate','basket','barrel','wagon','ladder','pillar','statue',
 'hedge','well','stove','cabinet','desk','sofa','stool','mirror','clock','vase','plant','bush','stump',
 'log','boulder','fountain','column','arch','tower','cabin','tent','cottage','garage','porch','pole','sign',
 'mailbox','trough','crib','kennel','coop','hive','nest','burrow',
 'lantern','bucket','trellis','planter','urn','birdbath','sundial','gazebo','pergola','archway',
 'wheelbarrow','toolshed','greenhouse','beehive','scarecrow','flagpole','signpost','milestone','culvert','drainpipe',
 'cistern','aqueduct','footbridge','stile','turnstile','railing','banister','balustrade','parapet','buttress',
 'chimney','rooftop','awning','canopy','veranda','balcony','staircase','doorway','threshold','alcove',
 'pallet','drum','sack','bin','vat','tub','jug','pail','keg','plank','beam',
 'rafter','joist','girder','strut','brace','truss','panel','board','curb','ledge','sill',
 'mantel','shelf','rack','hook','peg','knob','latch','hinge','bolt','nail','screw',
 'clamp','vice','anvil','forge','kiln','furnace','boiler','tank','pipe','valve','gauge',
 'dial','switch','socket','cable','wire','rope','chain','cord','strap','belt','buckle',
 'clasp','gravel','mound','heap','pile','mulch','compost','bale','haystack','silo','granary',
 'stable','paddock','pen','corral','pasture','meadow','orchard','vineyard','grove','thicket','bramble',
 'bracken','fern','reed','rush','sedge','moss','lichen','vine','ivy','creeper','shrub',
 'sapling','seedling','bulb','tuber','root','crag','cliff','ridge','slope','bank','dune',
 'mesa','butte','gorge','ravine','gully','ditch','furrow','trench','moat','embankment','levee',
 'dyke','weir','sluice','lock','dam','reservoir','crockery','platter','saucer','kettle','cauldron',
 'pot','pan','skillet','griddle','ladle','whisk','sieve','colander','funnel','jar','flask',
 'vial','beaker','tumbler','goblet','chalice',
]
DISTRACTORS = make_pairs(_DIST_SG)

PREPS = ["near the", "behind the", "beside the", "past the",
         "under the", "by the", "above the", "below the",
         "beneath the", "around the", "between the", "atop the",
         "outside the", "opposite the", "alongside the"]

_VERB = [
 ('jumps','jump'),('runs','run'),('eats','eat'),('sleeps','sleep'),('sings','sing'),('swims','swim'),
 ('dances','dance'),('plays','play'),('works','work'),('reads','read'),('waits','wait'),('rests','rest'),
 ('hides','hide'),('climbs','climb'),('digs','dig'),('hunts','hunt'),('feeds','feed'),('wanders','wander'),
 ('gathers','gather'),('returns','return'),('leaps','leap'),('crawls','crawl'),('floats','float'),
 ('glides','glide'),('wakes','wake'),('roams','roam'),('grazes','graze'),('naps','nap'),('barks','bark'),
 ('howls','howl'),('growls','growl'),('chirps','chirp'),('hops','hop'),('darts','dart'),('creeps','creep'),
 ('wades','wade'),('perches','perch'),('nests','nest'),('forages','forage'),('crouches','crouch'),
 ('pounces','pounce'),('scurries','scurry'),('burrows','burrow'),('paddles','paddle'),('drifts','drift'),
 ('circles','circle'),('soars','soar'),('dives','dive'),('waddles','waddle'),('stalks','stalk'),
 ('prowls','prowl'),('sniffs','sniff'),('licks','lick'),('scratches','scratch'),('shivers','shiver'),
 ('trembles','tremble'),('yawns','yawn'),('stretches','stretch'),('pauses','pause'),('lingers','linger'),
 ('settles','settle'),('stirs','stir'),
 ('hovers','hover'),('flaps','flap'),('flutters','flutter'),('swoops','swoop'),('plunges','plunge'),
 ('bounds','bound'),('gallops','gallop'),('trots','trot'),('canters','canter'),('ambles','amble'),
 ('strolls','stroll'),('saunters','saunter'),('shuffles','shuffle'),('slinks','slink'),('scampers','scamper'),
 ('scrambles','scramble'),('clambers','clamber'),('tumbles','tumble'),('rolls','roll'),('spins','spin'),
 ('twirls','twirl'),('sways','sway'),('rocks','rock'),('bobs','bob'),('nods','nod'),('blinks','blink'),
 ('snorts','snort'),('grunts','grunt'),('squeaks','squeak'),('squeals','squeal'),('hisses','hiss'),
 ('purrs','purr'),('mews','mew'),('bleats','bleat'),('clucks','cluck'),('quacks','quack'),('hoots','hoot'),
 ('croaks','croak'),('buzzes','buzz'),('chitters','chitter'),('whistles','whistle'),('whines','whine'),
 ('walks','walk'),('marches','march'),('splashes','splash'),('wallows','wallow'),('lurks','lurk'),('loiters','loiter'),
 ('idles','idle'),('dawdles','dawdle'),('meanders','meander'),('zigzags','zigzag'),('weaves','weave'),('bolts','bolt'),
 ('sprints','sprint'),('scuttles','scuttle'),('scoots','scoot'),('skitters','skitter'),('skips','skip'),('bounces','bounce'),
 ('vaults','vault'),('springs','spring'),('lunges','lunge'),('charges','charge'),('rushes','rush'),('races','race'),
 ('dashes','dash'),('hurries','hurry'),('hastens','hasten'),('flees','flee'),('scatters','scatter'),('disperses','disperse'),
 ('retreats','retreat'),('advances','advance'),('approaches','approach'),('departs','depart'),('arrives','arrive'),('remains','remain'),
 ('stays','stay'),('paces','pace'),('strides','stride'),('stomps','stomp'),('stamps','stamp'),('tramps','tramp'),
 ('trudges','trudge'),('plods','plod'),('lumbers','lumber'),('slogs','slog'),('limps','limp'),('hobbles','hobble'),
 ('staggers','stagger'),('teeters','teeter'),('wobbles','wobble'),('totters','totter'),('reels','reel'),('lurches','lurch'),
 ('sags','sag'),('slumps','slump'),('slouches','slouch'),('leans','lean'),('roosts','roost'),('squats','squat'),
 ('kneels','kneel'),('sprawls','sprawl'),('reclines','recline'),('lounges','lounge'),('dozes','doze'),('slumbers','slumber'),
 ('dreams','dream'),('drowses','drowse'),('rouses','rouse'),('awakens','awaken'),('arises','arise'),('rises','rise'),
 ('descends','descend'),('ascends','ascend'),('mounts','mount'),('scales','scale'),('tunnels','tunnel'),('roots','root'),
 ('rummages','rummage'),('probes','probe'),('pokes','poke'),('nudges','nudge'),('nuzzles','nuzzle'),('paws','paw'),
 ('claws','claw'),('scrapes','scrape'),('gnaws','gnaw'),('nibbles','nibble'),('chews','chew'),('munches','munch'),
 ('browses','browse'),('pecks','peck'),('sips','sip'),('laps','lap'),('gulps','gulp'),('swallows','swallow'),
 ('drinks','drink'),('feasts','feast'),('scavenges','scavenge'),('hoards','hoard'),('stashes','stash'),('chases','chase'),
 ('pursues','pursue'),('tracks','track'),('trails','trail'),('ambushes','ambush'),('snatches','snatch'),('seizes','seize'),
 ('grabs','grab'),('clutches','clutch'),('grasps','grasp'),('clasps','clasp'),('grips','grip'),('tugs','tug'),
 ('hauls','haul'),('drags','drag'),('heaves','heave'),('shoves','shove'),('pushes','push'),('thrusts','thrust'),
 ('swats','swat'),('flails','flail'),('thrashes','thrash'),('writhes','writhe'),('squirms','squirm'),('wriggles','wriggle'),
 ('twists','twist'),('coils','coil'),('curls','curl'),('uncoils','uncoil'),('unfurls','unfurl'),('flexes','flex'),
 ('arches','arch'),('hunches','hunch'),('bristles','bristle'),('shudders','shudder'),('quivers','quiver'),('quakes','quake'),
 ('flinches','flinch'),('recoils','recoil'),('cringes','cringe'),('cowers','cower'),('hunkers','hunker'),('huddles','huddle'),
 ('nestles','nestle'),('snuggles','snuggle'),('cuddles','cuddle'),('preens','preen'),('grooms','groom'),('bathes','bathe'),
 ('surfaces','surface'),('coasts','coast'),('sails','sail'),('skims','skim'),('flits','flit'),('wheels','wheel'),
 ('banks','bank'),('veers','veer'),('swerves','swerve'),('spirals','spiral'),('plummets','plummet'),
 ('strays','stray'),('roves','rove'),('ranges','range'),('migrates','migrate'),
 ('shelters','shelter'),('basks','bask'),('suns','sun'),
 ('caws','caw'),('coos','coo'),('trills','trill'),('warbles','warble'),
 ('chatters','chatter'),('clicks','click'),('drums','drum'),('thumps','thump'),('taps','tap'),
 ('rustles','rustle'),('scuffs','scuff'),('shambles','shamble'),('lopes','lope'),
 ('hurdles','hurdle'),('clears','clear'),
 ('slithers','slither'),('undulates','undulate'),('ripples','ripple'),('flows','flow'),
 ('streams','stream'),('pours','pour'),('trickles','trickle'),('seeps','seep'),('oozes','ooze'),
 ('tenses','tense'),('relaxes','relax'),('loosens','loosen'),('tightens','tighten'),
 ('clenches','clench'),('releases','release'),('drops','drop'),('lifts','lift'),
 ('raises','raise'),('lowers','lower'),('tilts','tilt'),('cranes','crane'),('cocks','cock'),
 ('swivels','swivel'),('pivots','pivot'),('rotates','rotate'),('revolves','revolve'),('orbits','orbit'),
]
VERBS = [{'sg':a,'pl':b} for a,b in dict.fromkeys(_VERB)]

COPULAS = [{'sg':'is','pl':'are'}]

_NOUN = [
 'duck','rabbit','frog','snake','lion','goat','bear','wolf','crab','hawk','toad','newt','mole','crow',
 'snail','moth','otter','badger','hare','vole','finch','wren','dove','swan','heron','robin','beetle',
 'spider','lizard','stoat','weasel','ferret','beaver','squirrel','gopher','turtle','gecko','iguana',
 'parrot','falcon','eagle','pigeon','quail','turkey','puffin','penguin','shrew','skunk','raccoon',
 'cricket','locust','hornet','aphid','firefly','ladybug','clam','oyster','prawn','shrimp','lobster',
 'eel','trout','carp','perch','pike','bass','cod','crane','stork','egret','plover','warbler','thrush',
 'raven','jay','cobra','viper','adder','boa','skink',
 'mouse','pony','zebra','camel','llama','donkey','mule','tiger','leopard','panda','bison','elk',
 'moose','mink','marten','gull','lark','sparrow','starling','magpie','kestrel','vulture','condor','pelican',
 'flamingo','tortoise','minnow','salmon','herring','sardine','shark','whale','walrus','seahorse','stingray','squid',
 'mantis','weevil','earwig','dragonfly','centipede','tick','flea','gnat','midge','termite','mussel','barnacle',
 'urchin','python','mamba','monitor','ibis','sandpiper','bunting','harrier','buzzard','jackdaw','rook','nuthatch',
 'tapir','gnu','ibex','wallaby','wombat','quokka','bandicoot','numbat','hedgehog','porcupine','possum','koala',
 'antelope','gazelle','cougar','jackal','coyote','dingo',
 'ox','ape','yak','bat','ray','boar','stag','deer','fawn','foal','joey','tern','kite',
 'slug','worm','grub','larva','sloth','lemur','hippo','rhino','sheep','hound','puppy',
 'piglet','kitten','cock','stallion','filly','heifer','steer','wether','hog',
 'gander','gosling','cygnet','duckling','fledgling','nestling','tadpole','guppy','koi',
 'cow','pig','hen','ewe','ram','lamb','calf','colt','mare','bull','dog','cat','bird','owl','fox',
]
NOUNS = make_pairs(_NOUN)

QUANT = {'sg':['one','every','a single'], 'pl':['many','several','two']}

ADJECTIVES = [
    'purple','tiny','clever','sleepy','angry','fuzzy','golden','spotted','silent','brave',
    'hungry','gentle','wild','striped','ancient','massive','small','large','little','big',
    'huge','giant','minor','major','vast','slight','broad','narrow','wide','thick',
    'thin','red','blue','green','yellow','orange','pink','brown','black','white',
    'gray','silver','bronze','crimson','scarlet','amber','fast','slow','quick','swift',
    'rapid','nimble','agile','sluggish','brisk','speedy','calm','quiet','loud','noisy',
    'fierce','tame','timid','bold','shy','meek','docile','feisty','happy','sad',
    'merry','gloomy','cheerful','glum','jolly','somber','joyful','weary','warm','cold',
    'cool','hot','chilly','frosty','icy','mild','balmy','tepid','soft','hard',
    'rough','smooth','bumpy','silky','furry','fluffy','coarse','sleek','bright','dark',
    'dim','dull','shiny','glossy','murky','pale','vivid','faded','young','old',
    'aged','youthful','elderly','mature','juvenile','strong','weak','sturdy','frail','robust',
    'feeble','mighty','puny','tough','fragile','clean','dirty','muddy','dusty','grimy',
    'spotless','filthy','tidy','messy','soiled','hairy','bald','scaly','feathered','woolly',
    'leathery','downy','bristly','shaggy','clumsy','graceful','elegant','awkward','lithe','ungainly',
    'dainty','curious','wary','alert','drowsy','restless','serene','placid','nervous','jittery',
    'edgy','lonely','social','friendly','hostile','savage','vicious','plump','skinny','lean',
    'stout','chubby','scrawny','portly','slender','lanky','burly','hidden','visible','secret',
    'obvious','sneaky','furtive','blatant','lazy','busy','idle','active','diligent','eager',
    'keen','foolish','wise','sharp','dense','astute','glowing','dewy','sandy','rocky',
    'leafy','mossy','grassy','weedy','crooked','straight','bent','curved','coiled','twisted',
    'gnarled','jagged','hollow','solid','airy','spongy','rigid','limp','stiff','supple',
    'loyal','fickle','steady','flighty','constant','erratic','dependable','wayward','mellow','harsh',
    'tender','brutal','kindly','cruel','humane','matted','tangled','groomed','ruffled','preened',
    'unkempt','wrinkled','creased','taut','saggy','firm','dotted','barred','banded','mottled',
    'dappled','flecked','freckled','velvety','satiny','rubbery','waxy','oily','greasy','fragrant',
    'musky','sour','sweet','bitter','pungent','acrid','earthy',
    'muscular','brawny','wiry','gaunt','haggard','rangy','hulking','strapping','willowy','svelte',
    'matte','velvet','woolen','plush','wispy','frizzy','curly','mangy','ragged','tattered',
    'threadbare','patched','worn','pristine','immaculate','flawless','radiant','luminous','gleaming','sparkling',
    'twinkling','shimmering','dazzling','glaring','blinding','glinting','shadowy','dusky','inky','pitch',
    'sooty','smoky','hazy','foggy','verdant','lush','flowering','blooming','withered','wilted',
    'parched','arid','barren','craggy','stony','pebbly','gravelly','silty','loamy','chalky',
    'clayey','marshy','boggy','bracing','crisp','nippy','raw','biting','piercing','numbing',
    'freezing','sultry','sweltering','scorching','blistering','searing','torrid','muggy','humid','steamy',
    'clammy','aromatic','perfumed','scented','odorous','rank','fetid','putrid','rancid','stale',
    'luscious','succulent','juicy','ripe','tangy','zesty','savory','bland','insipid','spry',
    'frisky','peppy','zippy','lively','sprightly','bouncy','jaunty','perky','languid','lethargic',
    'listless','torpid','logy','groggy','dozy','comatose','inert','motionless','skittish','jumpy',
    'twitchy','spooked','rattled','frazzled','frantic','frenzied','manic','stoic','composed','unflappable',
    'poised','collected','unruffled','aloof','detached','remote','affable','genial','cordial','amiable',
    'convivial','chummy','matey','neighborly','hospitable','warmhearted','surly','grumpy','crabby','cranky',
    'testy','peevish','irritable','petulant','sullen','morose','dapper','natty','spruce','trim',
    'suave','debonair','rakish','foppish','swanky','posh','dowdy','frumpy','shabby','scruffy',
    'slovenly','disheveled','bedraggled','rumpled','tousled','grubby','flexible','bendy','elastic','springy',
    'sinewy','prickly','thorny','barbed','spiny','needled','bristled','spiked','serrated','notched',
    'marbled','veined','streaked','striated','brindled','piebald','roan','opaque','translucent','transparent',
    'clear','cloudy','milky','frosted','tinted','shaded','glazed','buoyant','weightless','feathery',
    'gossamer','flimsy','insubstantial','ethereal','diaphanous','ponderous','leaden','hefty','bulky','cumbersome',
    'unwieldy','weighty','compact',
    'azure','teal','indigo','violet','maroon','olive','beige','tan','khaki','rust',
    'russet','ochre','umber','sienna','copper','brassy','golden','silvery','pearly','ivory',
    'creamy','snowy','ashen','slate','charcoal','ebony','jet','coal','dun','fawn',
    'tawny','sallow','swarthy','ruddy','rosy','peachy','salmon','coral','cherry','wine',
    'plum','lilac','mauve','lavender','periwinkle','turquoise','aqua','cyan','emerald','jade',
    'mint','lime','chartreuse','mustard','honey','caramel','toffee','cocoa','mahogany','walnut',
    'hazel','blond','flaxen','platinum','ginger','auburn','speckled','blotchy','splotchy','patchy',
    'pied','calico','tabby','tortoise','spangled','stippled','dotty','ringed','collared','crested',
    'tufted','plumed','maned','horned','antlered','tusked','clawed','fanged','beaked','webbed',
    'hoofed','pawed','tailed','whiskered','furred','plumose','quilled','spurred','taloned','snouted',
    'broad','squat','rotund','globular','bulbous','tubby','rotten','squidgy','squishy','doughy',
    'pudgy','roly','dumpy','blubbery','flabby','jiggly','wobbly','quivery','shaky','trembly',
    'fluttery','flighty','flitty','darting','dashing','bounding','leaping','prancing','cavorting','romping',
    'frolicsome','playful'
]
RC_PRONOUNS = ["he","she","they","I","we","you","someone","everyone","nobody","people"]
RC_VERBS = ["saw","chased","found","fed","held","raised","caught","watched","named","loved",
            "trained","kept","freed","led","spotted","tracked","hunted","tamed","walked","groomed",
            "fetched","carried","cornered","followed","studied","filmed","counted","petted",
            "leashed","penned","herded","adopted","rescued","bought","sold","bred","sketched"]
SG_DEMS = ["this","that"]
PL_DEMS = ["these","those","some"]

ATTRACTORS = [0, 1, 2, 3]
N_FRAMES   = 44

# ============================================================
#  Tokenization filter FIRST, then GLOBAL 50/50 split OVER PAIRS.
#  Change vs. the per-form version:
#    * sg/pl banks (subjects, nouns, verbs, distractors) are split
#      as whole LEMMAS keyed by the sg form, so a word's singular
#      and plural ALWAYS land on the same side. This is what makes
#      a merged sg+pl tuple coherent and guarantees a given
#      number-pair can appear only in train OR test.
#    * single-form content (adjectives, rc_verbs, prons, prep heads)
#      is still split per token.
#  Closed-class scaffolding is shared, never split:
#    'the', is/are, demonstratives, quantifiers, 'hold'.
# ============================================================
rng = random.Random(0)

# ---- tokenizer + single-token filter (BEFORE splitting) ----
def _tokenizer(lm):
    candidates = [
        getattr(lm, "tokenizer", None),
        getattr(getattr(lm, "processor", None), "tokenizer", None),
        getattr(getattr(lm, "tokenizer", None), "tokenizer", None),
    ]
    for c in candidates:
        if c is not None and hasattr(c, "encode"):
            return c
    raise AttributeError("Could not find a tokenizer with .encode on lm")

TOK = _tokenizer(lm)
print(type(TOK).__name__, "| has encode:", hasattr(TOK, "encode"))

def n_tokens(word, leading_space=True):
    s = (" " + word) if leading_space else word
    return len(TOK.encode(s, add_special_tokens=False))

def is_single_token(word):
    return n_tokens(word) == 1

def filter_pairs(bank):            # sg/pl dict banks: keep only if BOTH forms single-token
    return [d for d in bank if is_single_token(d["sg"]) and is_single_token(d["pl"])]

VERBS       = filter_pairs(VERBS)
COPULAS     = filter_pairs(COPULAS)
NOUNS       = filter_pairs(NOUNS)
SUBJECTS    = filter_pairs(SUBJECTS)
DISTRACTORS = filter_pairs(DISTRACTORS)
ADJECTIVES  = [a for a in ADJECTIVES  if is_single_token(a)]
RC_VERBS    = [v for v in RC_VERBS    if is_single_token(v)]
RC_PRONOUNS = [p for p in RC_PRONOUNS if is_single_token(p)]
print(f"after single-token filter: {len(VERBS)} verbs, {len(COPULAS)} copulas, "
      f"{len(NOUNS)} nouns, {len(SUBJECTS)} subjects, {len(DISTRACTORS)} distractors, "
      f"{len(ADJECTIVES)} adjectives, {len(RC_VERBS)} rc_verbs, {len(RC_PRONOUNS)} prons")

# ---- closed-class scaffolding: shared, never assigned to a split ----
SHARED_FUNC = {'the', 'is', 'are', 'hold',
               'this', 'that', 'these', 'those', 'some',
               'one', 'every', 'a', 'single', 'many', 'several', 'two'}

def prep_head(p):
    toks = [t for t in re.findall(r"[A-Za-z]+", p.lower()) if t not in SHARED_FUNC]
    return toks[0] if toks else p.lower()

# ---- build the split assignment, keyed by SURFACE STRING ----
#  Every content string is assigned to exactly one side. A pair
#  contributes BOTH its forms (sg & pl) as strings that must share
#  a side, so a number-pair can never straddle the split. Strings
#  are deduped GLOBALLY across banks, so a homograph that appears
#  in several banks (e.g. 'crane' = verb + subject, 'bank'/'nest'/
#  'root' = verb + distractor, 'idle' = adj + verb) travels as ONE
#  unit and cannot leak. We group the two forms of every pair into
#  one atomic unit (frozenset of its strings) before splitting, so
#  pl forms follow their sg even across collisions.
PAIR_BANKS = {'subj': SUBJECTS, 'noun': NOUNS, 'verb': VERBS, 'dist': DISTRACTORS}

# union-find over strings: any two strings that must co-travel
# (the sg & pl of one pair) are merged into the same component.
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[ra] = rb

all_strings = set()
for bank in PAIR_BANKS.values():
    for d in bank:
        union(d['sg'], d['pl'])
        all_strings.update([d['sg'], d['pl']])
for words in (ADJECTIVES, RC_VERBS, RC_PRONOUNS):
    for w in words:
        find(w); all_strings.add(w)
for p in PREPS:
    h = prep_head(p); find(h); all_strings.add(h)

# components = atomic split units
comps = {}
for s in all_strings:
    comps.setdefault(find(s), set()).add(s)
# sort components (by sorted member tuple) BEFORE shuffling so the
# split is fully reproducible across runs given the fixed seed.
units = [tuple(sorted(c)) for c in comps.values()]
units.sort()
rng.shuffle(units)
mid = len(units) // 2
TRAIN_VOCAB, TEST_VOCAB = set(), set()
for i, comp in enumerate(units):
    (TRAIN_VOCAB if i < mid else TEST_VOCAB).update(comp)
print(f"split units (string components): {len(units)}  ->  "
      f"train-strings {len(TRAIN_VOCAB)} / test-strings {len(TEST_VOCAB)}")

# ---- expand the global string assignment into per-bank pools -
#  A pair goes to a side iff its strings are in that side's vocab
#  (both always are, by construction). Single-form banks: by form.
def split_pairs(bank, bank_name=None):
    tr = [d for d in bank if d['sg'] in TRAIN_VOCAB]
    te = [d for d in bank if d['sg'] in TEST_VOCAB]
    return tr, te
def split_words(words, kind=None):
    tr = [w for w in words if w in TRAIN_VOCAB]
    te = [w for w in words if w in TEST_VOCAB]
    return tr, te
def split_preps(preps):
    tr = [p for p in preps if prep_head(p) in TRAIN_VOCAB]
    te = [p for p in preps if prep_head(p) in TEST_VOCAB]
    return tr, te

SUBJ_TRAIN, SUBJ_TEST = split_pairs(SUBJECTS, 'subj')
NOUN_TRAIN, NOUN_TEST = split_pairs(NOUNS, 'noun')
VERB_TRAIN, VERB_TEST = split_pairs(VERBS, 'verb')
DIST_TRAIN, DIST_TEST = split_pairs(DISTRACTORS, 'dist')
ADJ_TRAIN,  ADJ_TEST  = split_words(ADJECTIVES, 'adj')
RCV_TRAIN,  RCV_TEST  = split_words(RC_VERBS, 'rcv')
PRON_TRAIN, PRON_TEST = split_words(RC_PRONOUNS, 'pron')
PREP_TRAIN, PREP_TEST = split_preps(PREPS)

print("per-bank train/test sizes:")
for name, tr, te in [("subjects",SUBJ_TRAIN,SUBJ_TEST),("nouns",NOUN_TRAIN,NOUN_TEST),
                     ("verbs",VERB_TRAIN,VERB_TEST),("adjs",ADJ_TRAIN,ADJ_TEST),
                     ("preps",PREP_TRAIN,PREP_TEST),("dists",DIST_TRAIN,DIST_TEST),
                     ("rc_verbs",RCV_TRAIN,RCV_TEST),("prons",PRON_TRAIN,PRON_TEST)]:
    print(f"  {name:9s}: {len(tr)} / {len(te)}")

# ---- prove disjointness: no content form shared across splits ----
#  (now over BOTH forms of every pair, so this also proves a
#   number-pair never straddles the split.)
def forms(banks_pairs, banks_words, preps):
    s = set()
    for b in banks_pairs:
        for d in b: s.update([d['sg'], d['pl']])
    for b in banks_words:
        s.update(b)
    for p in preps: s.add(prep_head(p))
    return s
tr_forms = forms([SUBJ_TRAIN,NOUN_TRAIN,VERB_TRAIN,DIST_TRAIN],[ADJ_TRAIN,RCV_TRAIN,PRON_TRAIN],PREP_TRAIN)
te_forms = forms([SUBJ_TEST, NOUN_TEST, VERB_TEST, DIST_TEST ],[ADJ_TEST, RCV_TEST, PRON_TEST ],PREP_TEST)
print("content-form overlap train∩test:", len(tr_forms & te_forms), sorted(tr_forms & te_forms)[:20])

# ============================================================
#  Generate real-animal stimuli from the split pools.
#  NOW: one row per (subject-frame) carries BOTH numbers as a
#  tuple of four sentences:
#     good_singular / bad_singular / good_plural / bad_plural
#  Singular first, then plural, everywhere.
#  Within a row the singular target takes PLURAL distractors and
#  the plural target takes SINGULAR distractors (dn = opposite of
#  target number), so each number gets its own base sentence.
# ============================================================
N_PER_COND = 700          # rows per (condition x attractor) cell, per split

def cap(s): return s[0].upper() + s[1:]
def cond(t, n, dn): return f"target_{t}_att{n}_{dn}"   # no num: row holds both numbers
def mk(*parts):
    return cap(re.sub(r"\s+", " ", " ".join(p for p in parts if p)).strip())
def _toklen(text):
    return len(TOK.encode(text, add_special_tokens=False))


def build_rows(subjects, nouns, verbs, adjectives, preps, distractors,
               rc_verbs, pronouns, split_name):
    def pp_chain(offset, n, dist_num):
        return " ".join(
            f"{preps[(offset+j) % len(preps)]} {distractors[(offset+j) % len(distractors)][dist_num]}"
            for j in range(n)
        )
    rows = []
    for si, subj in enumerate(subjects):
        for n in ATTRACTORS:
            for f in range(N_FRAMES):
                offset = si + f
                # distractor number for each target number = OPPOSITE of target
                dn_sg, dn_pl = 'pl', 'sg'   # sg target -> pl distractors; pl target -> sg distractors

                # ---------------- VERB ----------------
                v = verbs[(si * 7 + f * 3) % len(verbs)]
                if n == 0:
                    adj = adjectives[(si + f) % len(adjectives)]
                    base_sg = mk("The", adj, subj['sg'])
                    base_pl = mk("The", adj, subj['pl'])
                    rows.append({
                        'split':split_name,'target_type':'verb','attractors':0,
                        'condition':cond('verb',0,'opp'),
                        'subject_word_sg': subj['sg'], 'subject_word_pl': subj['pl'],
                        'target_word_sg': v['sg'], 'target_word_pl': v['pl'],
                        'base_sentence_sg': base_sg, 'base_sentence_pl': base_pl,
                        'good_singular': mk("The", adj, subj['sg'], v['sg']),
                        'bad_singular':  mk("The", adj, subj['sg'], v['pl']),
                        'good_plural':   mk("The", adj, subj['pl'], v['pl']),
                        'bad_plural':    mk("The", adj, subj['pl'], v['sg']),
                    })
                else:
                    pp_sg = pp_chain(offset, n, dn_sg)
                    pp_pl = pp_chain(offset, n, dn_pl)
                    rows.append({
                        'split':split_name,'target_type':'verb','attractors':n,
                        'condition':cond('verb',n,'opp'),
                        'subject_word_sg': subj['sg'], 'subject_word_pl': subj['pl'],
                        'target_word_sg': v['sg'], 'target_word_pl': v['pl'],
                        'base_sentence_sg': mk("The", subj['sg'], pp_sg),
                        'base_sentence_pl': mk("The", subj['pl'], pp_pl),
                        'good_singular': mk("The", subj['sg'], pp_sg, v['sg']),
                        'bad_singular':  mk("The", subj['sg'], pp_sg, v['pl']),
                        'good_plural':   mk("The", subj['pl'], pp_pl, v['pl']),
                        'bad_plural':    mk("The", subj['pl'], pp_pl, v['sg']),
                    })

                # ---------------- COPULA ----------------
                c = COPULAS[0]
                if n == 0:
                    adj_c = adjectives[(si * 5 + f * 2 + 7) % len(adjectives)]
                    rows.append({
                        'split':split_name,'target_type':'copula','attractors':0,
                        'condition':cond('copula',0,'opp'),
                        'subject_word_sg': subj['sg'], 'subject_word_pl': subj['pl'],
                        'target_word_sg': c['sg'], 'target_word_pl': c['pl'],
                        'base_sentence_sg': mk("The", adj_c, subj['sg']),
                        'base_sentence_pl': mk("The", adj_c, subj['pl']),
                        'good_singular': mk("The", adj_c, subj['sg'], c['sg']),
                        'bad_singular':  mk("The", adj_c, subj['sg'], c['pl']),
                        'good_plural':   mk("The", adj_c, subj['pl'], c['pl']),
                        'bad_plural':    mk("The", adj_c, subj['pl'], c['sg']),
                    })
                else:
                    pp_sg = pp_chain(offset, n, dn_sg)
                    pp_pl = pp_chain(offset, n, dn_pl)
                    rows.append({
                        'split':split_name,'target_type':'copula','attractors':n,
                        'condition':cond('copula',n,'opp'),
                        'subject_word_sg': subj['sg'], 'subject_word_pl': subj['pl'],
                        'target_word_sg': c['sg'], 'target_word_pl': c['pl'],
                        'base_sentence_sg': mk("The", subj['sg'], pp_sg),
                        'base_sentence_pl': mk("The", subj['pl'], pp_pl),
                        'good_singular': mk("The", subj['sg'], pp_sg, c['sg']),
                        'bad_singular':  mk("The", subj['sg'], pp_sg, c['pl']),
                        'good_plural':   mk("The", subj['pl'], pp_pl, c['pl']),
                        'bad_plural':    mk("The", subj['pl'], pp_pl, c['sg']),
                    })

                # ---------------- NOUN ----------------
                tn = nouns[(si + f) % len(nouns)]
                if n == 0:
                    pron = pronouns[(si * 3 + f) % len(pronouns)]
                    rcv  = rc_verbs[(si + f * 2) % len(rc_verbs)]
                    sg_dem = SG_DEMS[f % len(SG_DEMS)]
                    pl_dem = PL_DEMS[f % len(PL_DEMS)]
                    rows.append({
                        'split':split_name,'target_type':'noun','attractors':0,
                        'condition':cond('noun',0,'opp'),
                        'subject_word_sg': tn['sg'], 'subject_word_pl': tn['pl'],
                        'target_word_sg': tn['sg'], 'target_word_pl': tn['pl'],
                        'base_sentence_sg': mk(cap(pron), rcv, sg_dem),
                        'base_sentence_pl': mk(cap(pron), rcv, pl_dem),
                        'good_singular': mk(cap(pron), rcv, sg_dem, tn['sg']),
                        'bad_singular':  mk(cap(pron), rcv, sg_dem, tn['pl']),
                        'good_plural':   mk(cap(pron), rcv, pl_dem, tn['pl']),
                        'bad_plural':    mk(cap(pron), rcv, pl_dem, tn['sg']),
                    })
                else:
                    pp_sg = pp_chain(offset, n, dn_sg)
                    pp_pl = pp_chain(offset, n, dn_pl)
                    q_sg = QUANT['sg'][f % len(QUANT['sg'])]
                    q_pl = QUANT['pl'][f % len(QUANT['pl'])]
                    rows.append({
                        'split':split_name,'target_type':'noun','attractors':n,
                        'condition':cond('noun',n,'opp'),
                        'subject_word_sg': tn['sg'], 'subject_word_pl': tn['pl'],
                        'target_word_sg': tn['sg'], 'target_word_pl': tn['pl'],
                        'base_sentence_sg': mk("The", subj['pl'], pp_sg, "hold", q_sg),
                        'base_sentence_pl': mk("The", subj['pl'], pp_pl, "hold", q_pl),
                        'good_singular': mk("The", subj['pl'], pp_sg, "hold", q_sg, tn['sg']),
                        'bad_singular':  mk("The", subj['pl'], pp_sg, "hold", q_sg, tn['pl']),
                        'good_plural':   mk("The", subj['pl'], pp_pl, "hold", q_pl, tn['pl']),
                        'bad_plural':    mk("The", subj['pl'], pp_pl, "hold", q_pl, tn['sg']),
                    })
    return rows


def finalize(rows, n_per_cell):
    # dedupe on the singular good form (its lexical frame uniquely keys the row)
    df = pd.DataFrame(rows).drop_duplicates(subset=['good_singular']).reset_index(drop=True)
    # token-length uniformity enforced on BOTH numbers together so the
    # sg and pl members of a kept row are length-matched within their column.
    df['tok_len_sg'] = df['good_singular'].map(_toklen)
    df['tok_len_pl'] = df['good_plural'].map(_toklen)
    keep_masks = []
    for ttype in df['target_type'].unique():
        for n in ATTRACTORS:
            cell_mask = (df['target_type'] == ttype) & (df['attractors'] == n)
            cell = df[cell_mask]
            if len(cell) == 0:
                continue
            mode_sg = cell['tok_len_sg'].mode().iloc[0]
            mode_pl = cell['tok_len_pl'].mode().iloc[0]
            keep_masks.append(cell_mask & (df['tok_len_sg'] == mode_sg)
                                        & (df['tok_len_pl'] == mode_pl))
    uniform = pd.concat([df[m] for m in keep_masks], ignore_index=False)
    pieces = []
    for ttype in uniform['target_type'].unique():
        for n in ATTRACTORS:
            cell = uniform[(uniform['target_type'] == ttype) & (uniform['attractors'] == n)]
            pieces.append(cell.sample(min(len(cell), n_per_cell), random_state=0))
    return pd.concat(pieces, ignore_index=True)


train_rows = build_rows(SUBJ_TRAIN, NOUN_TRAIN, VERB_TRAIN, ADJ_TRAIN,
                        PREP_TRAIN, DIST_TRAIN, RCV_TRAIN, PRON_TRAIN, "train")
test_rows  = build_rows(SUBJ_TEST,  NOUN_TEST,  VERB_TEST,  ADJ_TEST,
                        PREP_TEST,  DIST_TEST,  RCV_TEST,  PRON_TEST,  "test")

bal_train = finalize(train_rows, N_PER_COND)
bal_test  = finalize(test_rows,  N_PER_COND)

bal = pd.concat([bal_train, bal_test], ignore_index=True)
bal.insert(0, 'idx', range(len(bal)))

print("Grid (rows=condition, cols=attractors) — TRAIN:")
print(bal[bal.split=='train'].pivot_table(index='target_type', columns='attractors',
      values='idx', aggfunc='count', fill_value=0))
print("\nGrid — TEST:")
print(bal[bal.split=='test'].pivot_table(index='target_type', columns='attractors',
      values='idx', aggfunc='count', fill_value=0))
print("\nper split:", dict(bal.groupby('split').size()))
print("TOTAL rows:", len(bal),
      "| unique sg base sentences:", bal['base_sentence_sg'].nunique())

bal.to_csv("agreement_target_natural.csv", index=False)

# ============================================================
#  Wug-subject stimuli: subject animal -> [wug]/[wugs].
#  Surrounding real words drawn ONLY from the TEST-half pools
#  (disjoint from train). Every row labeled split="test".
#  Same tuple structure: good/bad x singular/plural.
# ============================================================
WUG_N_FRAMES = 200
WUG = {'sg': '[wug]', 'pl': '[wugs]'}

W_ADJ, W_VERB, W_PREP, W_DIST, W_RCV, W_PRON = (
    ADJ_TEST, VERB_TEST, PREP_TEST, DIST_TEST, RCV_TEST, PRON_TEST)

N_WUG_SLOTS = max(len(SUBJ_TEST), 30)

def wug_pp_chain(offset, n, dist_num):
    return " ".join(
        f"{W_PREP[(offset+j) % len(W_PREP)]} {W_DIST[(offset+j) % len(W_DIST)][dist_num]}"
        for j in range(n)
    )

rows_wug = []
for si in range(N_WUG_SLOTS):
    for n in ATTRACTORS:
        for f in range(WUG_N_FRAMES):
            offset = si + f
            dn_sg, dn_pl = 'pl', 'sg'
            ai  = (si * 5 + f * 2) % len(W_ADJ)
            aci = (si * 5 + f * 2 + 7) % len(W_ADJ)
            vi  = (si * 7 + f * 3) % len(W_VERB)

            # ---------------- VERB ----------------
            v = W_VERB[vi]
            if n == 0:
                adj = W_ADJ[ai]
                rows_wug.append({
                    'split':'test','target_type':'verb','attractors':0,
                    'condition':cond('verb',0,'opp'),
                    'subject_word_sg': WUG['sg'], 'subject_word_pl': WUG['pl'],
                    'target_word_sg': v['sg'], 'target_word_pl': v['pl'],
                    'base_sentence_sg': mk("The", adj, WUG['sg']),
                    'base_sentence_pl': mk("The", adj, WUG['pl']),
                    'good_singular': mk("The", adj, WUG['sg'], v['sg']),
                    'bad_singular':  mk("The", adj, WUG['sg'], v['pl']),
                    'good_plural':   mk("The", adj, WUG['pl'], v['pl']),
                    'bad_plural':    mk("The", adj, WUG['pl'], v['sg']),
                })
            else:
                pp_sg = wug_pp_chain(offset, n, dn_sg)
                pp_pl = wug_pp_chain(offset, n, dn_pl)
                rows_wug.append({
                    'split':'test','target_type':'verb','attractors':n,
                    'condition':cond('verb',n,'opp'),
                    'subject_word_sg': WUG['sg'], 'subject_word_pl': WUG['pl'],
                    'target_word_sg': v['sg'], 'target_word_pl': v['pl'],
                    'base_sentence_sg': mk("The", WUG['sg'], pp_sg),
                    'base_sentence_pl': mk("The", WUG['pl'], pp_pl),
                    'good_singular': mk("The", WUG['sg'], pp_sg, v['sg']),
                    'bad_singular':  mk("The", WUG['sg'], pp_sg, v['pl']),
                    'good_plural':   mk("The", WUG['pl'], pp_pl, v['pl']),
                    'bad_plural':    mk("The", WUG['pl'], pp_pl, v['sg']),
                })

            # ---------------- COPULA ----------------
            c = COPULAS[0]
            if n == 0:
                adj_c = W_ADJ[aci]
                rows_wug.append({
                    'split':'test','target_type':'copula','attractors':0,
                    'condition':cond('copula',0,'opp'),
                    'subject_word_sg': WUG['sg'], 'subject_word_pl': WUG['pl'],
                    'target_word_sg': c['sg'], 'target_word_pl': c['pl'],
                    'base_sentence_sg': mk("The", adj_c, WUG['sg']),
                    'base_sentence_pl': mk("The", adj_c, WUG['pl']),
                    'good_singular': mk("The", adj_c, WUG['sg'], c['sg']),
                    'bad_singular':  mk("The", adj_c, WUG['sg'], c['pl']),
                    'good_plural':   mk("The", adj_c, WUG['pl'], c['pl']),
                    'bad_plural':    mk("The", adj_c, WUG['pl'], c['sg']),
                })
            else:
                pp_sg = wug_pp_chain(offset, n, dn_sg)
                pp_pl = wug_pp_chain(offset, n, dn_pl)
                rows_wug.append({
                    'split':'test','target_type':'copula','attractors':n,
                    'condition':cond('copula',n,'opp'),
                    'subject_word_sg': WUG['sg'], 'subject_word_pl': WUG['pl'],
                    'target_word_sg': c['sg'], 'target_word_pl': c['pl'],
                    'base_sentence_sg': mk("The", WUG['sg'], pp_sg),
                    'base_sentence_pl': mk("The", WUG['pl'], pp_pl),
                    'good_singular': mk("The", WUG['sg'], pp_sg, c['sg']),
                    'bad_singular':  mk("The", WUG['sg'], pp_sg, c['pl']),
                    'good_plural':   mk("The", WUG['pl'], pp_pl, c['pl']),
                    'bad_plural':    mk("The", WUG['pl'], pp_pl, c['sg']),
                })

            # ---------------- NOUN (target = wug) ----------------
            if n == 0:
                pron = W_PRON[(si * 3 + f) % len(W_PRON)]
                rcv  = W_RCV[(si + f * 2) % len(W_RCV)]
                sg_dem = SG_DEMS[f % len(SG_DEMS)]
                pl_dem = PL_DEMS[f % len(PL_DEMS)]
                rows_wug.append({
                    'split':'test','target_type':'noun','attractors':0,
                    'condition':cond('noun',0,'opp'),
                    'subject_word_sg': WUG['sg'], 'subject_word_pl': WUG['pl'],
                    'target_word_sg': WUG['sg'], 'target_word_pl': WUG['pl'],
                    'base_sentence_sg': mk(cap(pron), rcv, sg_dem),
                    'base_sentence_pl': mk(cap(pron), rcv, pl_dem),
                    'good_singular': mk(cap(pron), rcv, sg_dem, WUG['sg']),
                    'bad_singular':  mk(cap(pron), rcv, sg_dem, WUG['pl']),
                    'good_plural':   mk(cap(pron), rcv, pl_dem, WUG['pl']),
                    'bad_plural':    mk(cap(pron), rcv, pl_dem, WUG['sg']),
                })
            else:
                pp_sg = wug_pp_chain(offset, n, dn_sg)
                pp_pl = wug_pp_chain(offset, n, dn_pl)
                q_sg = QUANT['sg'][f % len(QUANT['sg'])]
                q_pl = QUANT['pl'][f % len(QUANT['pl'])]
                rows_wug.append({
                    'split':'test','target_type':'noun','attractors':n,
                    'condition':cond('noun',n,'opp'),
                    'subject_word_sg': WUG['sg'], 'subject_word_pl': WUG['pl'],
                    'target_word_sg': WUG['sg'], 'target_word_pl': WUG['pl'],
                    'base_sentence_sg': mk("The", WUG['pl'], pp_sg, "hold", q_sg),
                    'base_sentence_pl': mk("The", WUG['pl'], pp_pl, "hold", q_pl),
                    'good_singular': mk("The", WUG['pl'], pp_sg, "hold", q_sg, WUG['sg']),
                    'bad_singular':  mk("The", WUG['pl'], pp_sg, "hold", q_sg, WUG['pl']),
                    'good_plural':   mk("The", WUG['pl'], pp_pl, "hold", q_pl, WUG['pl']),
                    'bad_plural':    mk("The", WUG['pl'], pp_pl, "hold", q_pl, WUG['sg']),
                })

bal_wug = finalize(rows_wug, N_PER_COND)
bal_wug.insert(0, 'idx', range(len(bal_wug)))

# --- sanity: no wug content word appears in the real-word TRAIN set ---
train_forms = set()
for col in ('good_singular', 'good_plural'):
    for s in bal[bal.split == 'train'][col]:
        train_forms.update(re.findall(r"[A-Za-z]+", s.lower()))
FUNC = {'the','a','this','that','these','those','some','one','every','single',
        'many','several','two','hold','is','are'}
wug_forms = set()
for col in ('good_singular', 'good_plural'):
    for s in bal_wug[col]:
        for w in re.findall(r"[A-Za-z]+", s.lower()):
            if w not in FUNC and w not in ('wug','wugs'):
                wug_forms.add(w)
leak = wug_forms & train_forms
print("wug content words leaking into TRAIN:", len(leak), sorted(leak)[:20])

print("\nGrid (rows=condition, cols=attractors) — WUG:")
print(bal_wug.pivot_table(index='target_type', columns='attractors',
      values='idx', aggfunc='count', fill_value=0))
print("\nunique sg base sentences (wug):")
print(bal_wug.groupby('target_type')['base_sentence_sg'].nunique())
print("TOTAL rows:", len(bal_wug),
      "| unique sg base sentences:", bal_wug['base_sentence_sg'].nunique())

bal_wug.to_csv("agreement_target_wug.csv", index=False)

Qwen2Tokenizer | has encode: True
after single-token filter: 94 verbs, 1 copulas, 29 nouns, 38 subjects, 88 distractors, 287 adjectives, 30 rc_verbs, 10 prons
split units (string components): 550  ->  train-strings 379 / test-strings 384
per-bank train/test sizes:
  subjects : 22 / 16
  nouns    : 18 / 11
  verbs    : 43 / 51
  adjs     : 150 / 137
  preps    : 6 / 9
  dists    : 41 / 47
  rc_verbs : 14 / 16
  prons    : 4 / 6
content-form overlap train∩test: 0 []
Grid (rows=condition, cols=attractors) — TRAIN:
attractors     0    1    2    3
target_type                    
copula       700  700  700  700
noun         504  660  660  660
verb         700  700  700  700

Grid — TEST:
attractors     0    1    2    3
target_type                    
copula       700  700  700  700
noun         640  480  480  480
verb         700  700  700  700

per split: {'test': np.int64(7680), 'train': np.int64(8084)}
TOTAL rows: 15764 | unique sg base sentences: 10369
wug content words leaking into TRAI